In [3]:
# Neural Machine Translation (English → French) using Helsinki-NLP MarianMT

# 1. Install required libraries (uncomment if needed)
# !pip install transformers datasets sentencepiece sacrebleu torch matplotlib

# 2. Import Libraries
import matplotlib.pyplot as plt
from datasets import load_dataset, DatasetDict
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import sacrebleu

# 3. Load Dataset
dataset = load_dataset("opus100", "en-fr")

# Flatten translation dict to separate columns
for split in ["train", "test"]:
    dataset[split] = dataset[split].map(lambda x: {"en": x["translation"]["en"], "fr": x["translation"]["fr"]})

# 4. Create Validation Split (5% of train)
train_valid = dataset["train"].train_test_split(test_size=0.05, seed=42)
dataset = DatasetDict({
    "train": train_valid["train"],
    "validation": train_valid["test"],
    "test": dataset["test"]
})

# 5. Load Pre-trained MarianMT Model and Tokenizer
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# 6. Preprocessing Function
def preprocess_function(examples):
    inputs = tokenizer(examples["en"], max_length=128, truncation=True, padding="max_length")
    targets = tokenizer(examples["fr"], max_length=128, truncation=True, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=["en","fr","translation"] if "translation" in dataset["train"].column_names else ["en","fr"])

# 7. Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,  # change to 3 or more for real training
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_strategy="epoch"
)

# 8. Data Collator for Dynamic Padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 9. Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# 10. Fine-tuning (uncomment to train)
# trainer.train()

# 11. BLEU Evaluation Example on small subset
test_subset = tokenized_dataset["test"].select(range(10))  # small subset for demo
preds = []
refs = []

for example in test_subset:
    inputs = tokenizer(example["labels"], return_tensors="pt", padding=True, truncation=True)
    translated_tokens = model.generate(inputs["input_ids"])
    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    preds.append(translated_text)
    refs.append([tokenizer.decode(example["labels"], skip_special_tokens=True)])

bleu = sacrebleu.corpus_bleu(preds, refs)
print("Sample Test BLEU score:", bleu.score)

Map:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

C:\Users\23adsb48\AppData\Roaming\Python\Python312\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/950000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [4]:
# Neural Machine Translation (English → French) using Helsinki-NLP MarianMT
# Using HuggingFace Transformers + Opus100

from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer
import sacrebleu
import torch

# 1. Load a small subset of Opus100 English-French dataset
dataset = load_dataset("opus100", "en-fr", split="test[:20]")  # first 20 examples

# Flatten translation dictionary to separate columns
src_texts = [ex["translation"]["en"] for ex in dataset]
refs = [[ex["translation"]["fr"]] for ex in dataset]  # BLEU expects list of references

# 2. Load pre-trained MarianMT model and tokenizer
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# 3. Tokenize input texts
inputs = tokenizer(src_texts, return_tensors="pt", padding=True, truncation=True, max_length=128)

# 4. Generate translations
with torch.no_grad():
    translated_tokens = model.generate(**inputs)

translated_texts = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

# 5. Print translations
print("\n--- Translations ---\n")
for src, pred, ref in zip(src_texts, translated_texts, refs):
    print("Source :", src)
    print("Pred   :", pred)
    print("Ref    :", ref[0])
    print("-"*60)

# 6. Compute BLEU score
bleu = sacrebleu.corpus_bleu(translated_texts, refs)
print("\nSample BLEU score:", bleu.score)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



--- Translations ---

Source : - You were at a bus stop kissing him!
Pred   : - Tu étais à un arrêt de bus pour l'embrasser !
Ref    : - Vous étiez en train de vous embrasser à l'arrêt de bus!
------------------------------------------------------------
Source : With irony and mischief the young Czech artist Krištof Kintera turns art and life on their heads.
Pred   : Avec l'ironie et le malice, la jeune artiste tchèque Krištof Kintera tourne l'art et la vie sur leur tête.
Ref    : Avec une ironie farceuse, le jeune artiste tchèque Krištof Kintera chamboule l’art et la vie.
------------------------------------------------------------
Source : - Who's going to talk to the used car salesman?
Pred   : - Qui va parler au vendeur de voitures d'occasion ?
Ref    : Qui va parler au vendeur de voitures ? Big Freddy.
------------------------------------------------------------
Source : People think you are a great man, that your music speaks of humanity, warmth and understanding.
Pred   : Les g